# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is published in Croissant format and can be accessed from its schema URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs as described by the schema.

We enumerate the record sets, then list their fields and columns by their `@id`.

In [ ]:
# Get record sets from metadata
record_sets = metadata.recordSet

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rset in record_sets:
        print(f"\nRecord Set: {rset['@id']}")
        print(f"  Name: {rset.get('name', 'N/A')}")
        print(f"  Description: {rset.get('description', 'N/A')}")
        
        # List fields and columns
        fields = rset.get('field', [])
        if fields:
            print("  Fields:")
            for f in fields:
                fid = f['@id'] if isinstance(f, dict) and '@id' in f else f
                print(f"    - {fid}")
        columns = rset.get('column', [])
        if columns:
            print("  Columns:")
            for col in columns:
                cid = col['@id'] if isinstance(col, dict) and '@id' in col else col
                print(f"    - {cid}")
        print()

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis.

**Note:** All referencing of record sets and fields uses their `@id`.

In [ ]:
# Retrieve all record set @ids
record_set_ids = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if isinstance(rs, dict) and '@id' in rs:
            record_set_ids.append(rs['@id'])
        elif isinstance(rs, str):
            record_set_ids.append(rs)

if not record_set_ids:
    print("No record sets available for extraction.")
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        # Load records into DataFrame
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record set {record_set_id} with shape {dataframes[record_set_id].shape}")
        else:
            print(f"No records returned for record set {record_set_id}.")
    
    # For demonstration, show columns and head of the first available record set
    if dataframes:
        demo_rsid = list(dataframes.keys())[0]
        print(f"\nDataFrame columns for {demo_rsid}:")
        print(dataframes[demo_rsid].columns.tolist())
        display(dataframes[demo_rsid].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data cleaning and analysis steps: filtering numeric fields, normalization, and grouping by key fields.

Again, all fields are referenced by their `@id`. Replace placeholders below based on the available fields and your data context.

In [ ]:
# Example: choose one available record set for EDA
if not dataframes:
    print("No DataFrames available for EDA.")
else:
    rsid = list(dataframes.keys())[0]
    df = dataframes[rsid]
    print(f"Examining record set: {rsid}. Columns: {df.columns.tolist()}")

    # Try to pick a numeric field by inspecting dtypes
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"\nUsing numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field]).mean() else 10

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f}")

        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"First 5 rows with normalized {numeric_field}:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Group by the first non-numeric field if available
        non_numeric_candidates = [c for c in df.columns if c != numeric_field and not pd.api.types.is_numeric_dtype(df[c])]
        if non_numeric_candidates:
            group_field = non_numeric_candidates[0]
            print(f"\nGrouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            display(grouped_df.head())
        else:
            print("No non-numeric field found for grouping.")
    else:
        print("No numeric fields found in record set for EDA.")

## 5. Visualization
Visualize a numeric field's distribution and group comparisons. All references use `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for the numeric field, if available
if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20, color='steelblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouping, show boxplot
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook provided a walkthrough for accessing, exploring, and visualizing the FAIR² rangeland management dataset using `mlcroissant`. 

Key steps included reading Croissant metadata, inspecting record sets and their `@id`s, extracting tabular data for analysis, filtering and transforming numeric fields, and plotting key distributions or group comparisons.

For further, dataset-specific insights, refer to schema documentation, field definitions (by `@id`), and study context as linked in the dataset metadata.

Happy analyzing!